# 06 · De la predicció a l'acció: perfils de risc i campanyes

Els models (notebooks 02–05) diuen **qui** marxa i **quan**. Aquest notebook ho tradueix en
**accions**: es defineixen perfils de clients fàcils d'explicar al negoci, es mesura el seu risc i
s'hi associa una campanya de retenció concreta. Finalment, se simula si la campanya principal (pas a
contracte anual) surt a compte econòmicament.

- **Taxa de baixa** d'un perfil: percentatge de clients del perfil que han marxat (descriptiva).
- **Risc a 12 mesos** dels clients actius: probabilitat de marxar en els propers 12 mesos segons el
  model de Cox ($1 - S_{12}$, notebook 04). Té un horitzó temporal clar, a diferència de la
  probabilitat del classificador.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.precision", 3)
from src.preparacio import carregar_dades
from src.risc_temporal import generar_scores
from src.perfils import (taula_contracte_tram, resum_perfils, actius_perfil, reduccio_maxima,
                         simular_campanya, efecte_minim_rendible, ESCENARIS, PERFILS)
from src.grafics_perfils import fig_mapa_calor, fig_perfils, fig_campanya
from src.grafics import exportar
df = carregar_dades()
scores = generar_scores(df)

## On es concentra el risc

Taxa de baixa per tipus de contracte i tram d'antiguitat (en mesos).

In [2]:
taxa, n = taula_contracte_tram(df)
fig = fig_mapa_calor(taxa, n, "Taxa de baixa per contracte i antiguitat")
exportar(fig, "pf_mapa_calor"); fig.show()
taxa

tram,1–6,7–12,13–24,25–48,49–72
Contract,,,,,
Mensual,0.552,0.420,0.377,0.329,0.260
1 any,0.105,0.106,0.081,0.106,0.129
2 anys,0.000,0.000,0.000,0.022,0.033


**Lectura.** El risc es concentra gairebé tot en el **contracte mensual**: als primers 6 mesos hi
marxa el **55 %** dels clients, i encara amb 1–2 anys d'antiguitat el **38 %**. Amb contracte
d'1 any la taxa es manté entre el 8 % i el 13 % sigui quina sigui l'antiguitat, i amb 2 anys és
pràcticament nul·la (0–3 %).

## Perfils de risc i campanyes

Quatre perfils, cadascun comparat amb un grup de referència semblant que marxa menys. La
diferència entre tots dos suggereix la palanca de la campanya.

| Perfil | Definició | Referència | Campanya |
|---|---|---|---|
| Clients nous amb contracte mensual | Contracte mensual i 6 mesos o menys d'antiguitat. | mensuals amb més de 6 mesos | Programa de benvinguda |
| Clients mensuals consolidats | Contracte mensual i 12 mesos o més d'antiguitat. | contracte d'1 any amb 12 mesos o més | Pas a contracte anual |
| Fibra sense suport ni seguretat | Contracte mensual, fibra òptica i sense suport tècnic ni seguretat en línia. | mensuals amb fibra, suport tècnic i seguretat | Suport i seguretat inclosos |
| Pagament amb xec electrònic | Contracte mensual i pagament amb xec electrònic. | mensuals amb pagament automàtic (transferència o targeta) | Domiciliació amb incentiu |

Un mateix client pot pertànyer a més d'un perfil, de manera que les xifres no s'han de sumar.

In [3]:
resum = resum_perfils(df, scores)
fig = fig_perfils(resum, "Taxa de baixa de cada perfil vs. un grup de referència")
exportar(fig, "pf_perfils"); fig.show()
resum[["perfil", "taxa", "taxa_ref", "diferencia_pp", "actius", "risc_12m",
       "baixes_esperades_12m", "ingressos_risc_mes", "campanya"]]

,perfil,taxa,taxa_ref,diferencia_pp,actius,risc_12m,baixes_esperades_12m,ingressos_risc_mes,campanya
0,Clients nous amb contracte mensual,0.552,0.355,19.661,633,0.219,138.892,7152.433,Programa de benvinguda
1,Clients mensuals consolidats,0.338,0.115,22.233,1303,0.259,337.556,25914.297,Pas a contracte anual
2,Fibra sense suport ni seguretat,0.607,0.280,32.647,599,0.348,208.746,17708.045,Suport i seguretat inclosos
3,Pagament amb xec electrònic,0.537,0.335,20.249,856,0.309,264.276,19999.162,Domiciliació amb incentiu


**Lectura.**

| Perfil | Taxa de baixa (vs. referència) | Clients actius | Baixes esperades en 12 mesos | Ingressos mensuals en risc |
|---|---|---|---|---|
| Clients nous amb contracte mensual | 55 % (vs. 36 %) | 633 | ≈ 139 | ≈ 7.150 € |
| Clients mensuals consolidats | 34 % (vs. 12 %) | 1.303 | ≈ 338 | ≈ 25.900 € |
| Fibra sense suport ni seguretat | 61 % (vs. 28 %) | 599 | ≈ 209 | ≈ 17.700 € |
| Pagament amb xec electrònic | 54 % (vs. 33 %) | 856 | ≈ 264 | ≈ 20.000 € |

- El perfil amb **més taxa de baixa** és el de fibra sense suport ni seguretat: tenir tots dos
  serveis la redueix a menys de la meitat.
- El perfil amb **més ingressos en joc** és el dels mensuals consolidats: són molts clients i paguen
  una quota mitjana alta (≈ 70 €).
- La taxa històrica dels clients nous (55 %) és molt més alta que el risc a 12 mesos dels que encara
  hi són (≈ 22 %): molts dels nous que havien de marxar ja ho han fet. La taxa descriu el perfil; el
  risc a 12 mesos diu quants clients actuals es poden perdre.

## Simulador: campanya de pas a contracte anual

Oferta als **clients mensuals consolidats** actius: un descompte durant 12 mesos a canvi de passar a
contracte d'1 any.

- **Reducció màxima del risc:** $1 - \dfrac{\text{taxa (1 any, ≥ 12 mesos)}}{\text{taxa (mensual, ≥ 12 mesos)}}$.
  És un **màxim**: part de la diferència es deu al contracte (marxar té penalització) i part a **qui
  tria** el contracte anual (clients ja més fidels, autoselecció).
- **Escenari optimista:** tota la reducció és real. **Escenari prudent:** només la meitat.
- **Baixes evitades** = clients que accepten × risc a 12 mesos × reducció × efecte.
- **Ingressos salvats** = baixes evitades × quota mitjana × 12 mesos.
- **Cost del descompte** = clients que accepten × quota mitjana × descompte × 12 mesos. El cobren
  **tots** els que accepten, també els que no haurien marxat.

Supòsits: accepta el 20 % dels clients i el descompte és del 10 %.

In [4]:
actius = actius_perfil(df, scores, "mensuals_consolidats")
red = reduccio_maxima(df)
resultats = {esc: simular_campanya(actius, 0.20, 0.10, red, efecte) for esc, efecte in ESCENARIS.items()}
fig = fig_campanya(resultats, "Resultat econòmic de la campanya en 12 mesos (20 % accepten, 10 % de descompte)")
exportar(fig, "pf_campanya"); fig.show()
print(f"Reducció màxima del risc: {red:.1%}")
print(f"Efecte mínim perquè surti a compte: {efecte_minim_rendible(actius['risc_12m'].mean(), red, 0.10):.1%}")
pd.DataFrame(resultats).T[["accepten", "baixes_sense_campanya", "baixes_evitades",
                           "ingressos_salvats", "cost_descompte", "benefici"]].round(0)

Reducció màxima del risc: 65.9%
Efecte mínim perquè surti a compte: 58.6%


,accepten,baixes_sense_campanya,baixes_evitades,ingressos_salvats,cost_descompte,benefici
Optimista,261.0,68.0,45.0,37490.0,21973.0,15517.0
Prudent,261.0,68.0,22.0,18745.0,21973.0,-3228.0


**Lectura.**

- Dels 1.303 clients mensuals consolidats actius, el **26 %** marxarien en 12 mesos. Els clients
  d'1 any amb la mateixa antiguitat marxen un **66 %** menys.
- Si accepta el 20 % (261 clients) amb un 10 % de descompte:
  - **optimista:** s'eviten ≈ 45 baixes i la campanya guanya ≈ **15.500 €** en 12 mesos;
  - **prudent:** s'eviten ≈ 22 baixes i la campanya **perd** ≈ 3.200 €.
- La campanya només surt a compte si l'efecte real és com a mínim el **59 %** de l'efecte màxim. El
  problema és que el descompte el reben també els clients que no haurien marxat.

## Conclusions

- **On actuar:** el risc es concentra en el contracte mensual, sobretot els primers mesos.
- **Com actuar:** cada perfil té una palanca diferent: benvinguda per als clients nous, pas a contracte
  anual per als consolidats, serveis de suport per a la fibra i domiciliació del pagament.
- **Quan:** els primers mesos de vida del client (notebook 04), que és quan el programa de benvinguda
  té més sentit.
- **Recomanació:** abans d'aplicar la campanya d'anual a tots els clients, cal fer un **pilot amb
  grup de control**: una part dels clients rep l'oferta i una altra no. Comparant-los es mesura
  l'efecte real i, amb **uplift modeling**, es poden identificar els clients que canvien de decisió
  gràcies a l'oferta, que són als únics a qui val la pena fer el descompte.

**Limitacions.** Les taxes de baixa són descriptives (dataset transversal). La reducció del risc
s'estima comparant grups diferents, no amb un experiment. El simulador suposa que els que accepten
tenen el risc i la quota mitjans del perfil, i no inclou el cost de contactar els clients ni
l'efecte que la campanya pugui tenir més enllà dels 12 mesos.